In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("../data/train.csv")

# Basic checks
df.shape
df.head()

,ID_code,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,train_0,0,8.9255,-6.7863,11.9081,5.0930,11.4607,-9.2834,5.1187,18.6266,...,4.4354,3.9642,3.1364,1.6910,18.5227,-2.3978,7.8784,8.5635,12.7803,-1.0914
1,train_1,0,11.5006,-4.1473,13.8588,5.3890,12.3622,7.0433,5.6208,16.5338,...,7.6421,7.7214,2.5837,10.9516,15.4305,2.0339,8.1267,8.7889,18.3560,1.9518
2,train_2,0,8.6093,-2.7457,12.0805,7.8928,10.5825,-9.0837,6.9427,14.6155,...,2.9057,9.7905,1.6704,1.6858,21.6042,3.1417,-6.5213,8.2675,14.7222,0.3965
3,train_3,0,11.0604,-2.1518,8.9522,7.1957,12.5846,-1.8361,5.8428,14.9250,...,4.4666,4.7433,0.7178,1.4214,23.0347,-1.2706,-2.9275,10.2922,17.9697,-8.9996
4,train_4,0,9.8369,-1.4834,12.8746,6.6375,12.2772,2.4486,5.9405,19.2514,...,-1.4905,9.5214,-0.1508,9.1942,13.2876,-1.5121,3.9267,9.5031,17.9974,-8.8104


In [2]:
# Count how many samples in each class
df['target'].value_counts()

# Percentage of each class
df['target'].value_counts(normalize=True)

target
0    0.89951
1    0.10049
Name: proportion, dtype: float64

In [3]:
from sklearn.model_selection import train_test_split

# Features and target
X = df.drop(columns=['target', 'ID_code'])
y = df['target']

# Split (IMPORTANT: stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((160000, 200), (40000, 200))

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Initialize model
model = LogisticRegression(max_iter=1000)

# Train
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

d:\6th semester study\Regrerssion-Comparison\research\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [5]:
# Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1-score:", f1)
print("ROC-AUC:", roc)

Accuracy: 0.912925
Precision: 0.6762967826657912
Recall: 0.2562189054726368
F1-score: 0.37163990618798487
ROC-AUC: 0.8579453206452452


In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })



In [7]:
import pandas as pd

pd.DataFrame(results)

,Model,Accuracy,F1,ROC-AUC
0,Decision Tree,0.837925,0.207748,0.559682
1,Random Forest,0.899500,0.000000,0.819835
2,Gradient Boosting,0.901925,0.055836,0.829760


## 📊 Baseline Model Comparison

### Models Evaluated
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting

---

### Evaluation Strategy
The dataset is highly imbalanced (~90% class 0, ~10% class 1), so multiple metrics were used:

- **Accuracy**: Not reliable due to imbalance
- **F1-score**: Measures balance between precision and recall
- **ROC-AUC**: Primary metric for evaluating model performance

---

### Results Summary

- Gradient Boosting achieved the highest ROC-AUC (~0.83), indicating strong ability to distinguish between classes
- Random Forest achieved high accuracy but an F1-score of 0, meaning it predicted only the majority class
- Decision Tree showed poor ROC-AUC, indicating weak generalization

---

### Key Observations

- **Severe class imbalance caused models to favor the majority class**
- Some models (e.g., Random Forest) failed to detect any minority class samples
- **ROC-AUC remained relatively high**, suggesting models learned useful patterns but failed at classification threshold

---

### Critical Insight

- Default classification threshold (0.5) is not suitable for imbalanced datasets
- High accuracy can be misleading and does not reflect true model performance
- There is a clear need to improve recall and F1-score for the minority class

---

### Next Steps

To improve performance:
- Apply techniques to handle class imbalance
- Adjust decision thresholds
- Explore better feature representation
